<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 07. PPO

## Proximal Policy Optimization Algorithms

- **원 논문:** [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347)
- **저자 / 발표:** John Schulman et al. · arXiv (2017)
- **난이도 / 예상 시간:** 중급 · 약 60분
- **선수 지식:** policy ratio, advantage, actor-critic
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 고정 old log-prob batch에서 clipped surrogate를 여러 epoch 최적화해 로컬 optimal policy를 학습한다.

**축소하거나 생략한 부분:** 논문은 MuJoCo/Atari에서 parallel actors와 GAE를 사용한다. 축소판은 exact q_star advantage를 써 clipping mechanics를 격리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (1) and Eq. (6) | policy-gradient/CPI surrogate | old-policy importance ratio objective의 출발점을 계산한다. |
| §3, Eq. (7) | clipped surrogate | min(rA, clip(r)A)로 큰 policy step의 이득을 제한한다. |
| §5, Eq. (9), Algorithm 1 | combined actor/value/entropy multi-epoch update | 같은 rollout batch를 여러 epoch 재사용한다. |

## 핵심 재현

        Eq. (7)의 $\min(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)$를
        구현합니다. old log-prob는 rollout 뒤 고정하고 같은 batch를 여러 epoch 사용합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(471)
np.random.seed(471)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
L^{CLIP}=\mathbb E_t\left[\min(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)\right]
$$

- **기호와 역할:** $r_t=\pi_\theta/\pi_{old}$는 probability ratio, $A_t$는 advantage, $\epsilon$은 policy step 제한입니다.
- **shape/state 계약:** `state [B] → policy [B, A], value [B]; logp/advantage/return [B]`
- **논문 위치:** `§2, Eq. (1) and Eq. (6)`의 **policy-gradient/CPI surrogate**
- **코드 Task:** old log-prob 고정, clipped surrogate, value/entropy 결합 update를 구현합니다.
- **학습·평가 흐름:** old-policy rollout → advantage normalize → multi-epoch clip update → policy 평가
- **원문 대비 한계:** 논문은 MuJoCo/Atari에서 parallel actors와 GAE를 사용한다. 축소판은 exact q_star advantage를 써 clipping mechanics를 격리한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class ProximalPolicyAgent(nn.Module):
    """PPO의 핵심 학습·평가 경로. 입출력 계약: state [B] → policy [B, A], value [B];
    logp/advantage/return [B]."""

    def __init__(self, logits, value_table, optimizer, clip_epsilon=0.2):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.logits = logits
        self.value_table = value_table
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"clip_epsilon": self.clip_epsilon, "epochs_per_batch": 4}

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: state [B] → policy [B, A], value [B];
        logp/advantage/return [B]."""
        probability = torch.softmax(self.logits[state], dim=1)
        value = self.value_table[state]
        return (probability, value)

    def loss(self, state, action, old_log_probability, advantage, returns):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        probability, value = self.forward(state)
        row = torch.arange(len(state), device=probability.device)
        new_log_probability = torch.log(probability[row, action].clamp_min(1e-08))
        ratio = torch.exp(new_log_probability - old_log_probability)
        unclipped = ratio * advantage
        clipped_ratio = ratio.clamp(1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon)
        clipped = clipped_ratio * advantage
        policy_objective = torch.minimum(unclipped, clipped).mean()
        value_loss = F.mse_loss(value, returns)
        entropy = (
            -(probability * torch.log(probability.clamp_min(1e-08))).sum(dim=1).mean()
        )
        return -policy_objective + 0.5 * value_loss - 0.01 * entropy

    def update(self, state, action, old_log_probability, advantage, returns):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(state, action, old_log_probability, advantage, returns)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def evaluate(self, reference_q):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            probability = torch.softmax(self.logits, dim=1)
            nonterminal = torch.arange(n_states) != terminal_state
            agreement = (
                (
                    probability.argmax(dim=1)[nonterminal]
                    == reference_q.argmax(dim=1)[nonterminal]
                )
                .float()
                .mean()
            )
            entropy = (
                -(probability * torch.log(probability.clamp_min(1e-08)))
                .sum(dim=1)
                .mean()
            )
        return {"greedy_agreement": float(agreement), "entropy": float(entropy)}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3, Eq. (7)` — **clipped surrogate**
- **구현 이유:** min(rA, clip(r)A)로 큰 policy step의 이득을 제한한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def ppo_objective(new_logp, old_logp, adv, eps=0.2):
    ratio = (new_logp - old_logp).exp()
    surrogate = torch.minimum(ratio * adv, ratio.clamp(1 - eps, 1 + eps) * adv)
    return surrogate.mean(), float(((ratio - 1).abs() > eps).float().mean())


obj, frac = ppo_objective(
    torch.log(torch.tensor([1.4, 0.8])),
    torch.zeros(2),
    torch.tensor([1.0, -1.0]),
)
assert obj.ndim == 0 and 0 <= frac <= 1

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§5, Eq. (9), Algorithm 1` — **combined actor/value/entropy multi-epoch update**
- **구현 이유:** 같은 rollout batch를 여러 epoch 재사용한다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
# The policy/value tables contain only a few hundred scalars and every
# iteration samples with a CPU generator, so this toy PPO fit is CPU-fast.
policy_logits = nn.Parameter(torch.zeros(n_states, n_actions))
values = nn.Parameter(torch.zeros(n_states))
opt = torch.optim.Adam([policy_logits, values], lr=0.04)
gen = torch.Generator().manual_seed(472)
history = []
batch_states = torch.arange(n_states).repeat_interleave(12)

portfolio_agent = ProximalPolicyAgent(policy_logits, values, opt)
initial_parameters = policy_logits.detach().clone()
for _ in range(45):
    with torch.no_grad():
        old_probability, old_value = portfolio_agent.forward(batch_states)
        batch_actions = torch.multinomial(old_probability, 1, generator=gen).squeeze(1)
        rows = torch.arange(len(batch_states))
        old_log_probability = torch.log(
            old_probability[rows, batch_actions].clamp_min(1e-8)
        )
        returns = q_star[batch_states, batch_actions]
        advantage = returns - old_value
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-6)
    epoch_losses = []
    for _ in range(4):
        epoch_losses.append(
            portfolio_agent.update(
                batch_states,
                batch_actions,
                old_log_probability,
                advantage,
                returns,
            )
        )
    with torch.no_grad():
        new_probability, _ = portfolio_agent.forward(batch_states)
        new_log_probability = torch.log(
            new_probability[rows, batch_actions].clamp_min(1e-8)
        )
        ratio = torch.exp(new_log_probability - old_log_probability)
        clip_fraction = float(((ratio - 1.0).abs() > 0.2).float().mean())
    history.append((epoch_losses[-1], clip_fraction))
parameter_delta = float((policy_logits.detach() - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§2, Eq. (1) and Eq. (6)` — **policy-gradient/CPI surrogate**
- **구현 이유:** old-policy importance ratio objective의 출발점을 계산한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(q_star)
assert portfolio_metrics

probs = policy_logits.softmax(1).detach()
nonterminal = torch.arange(n_states) != terminal_state
agreement = float(
    (probs.argmax(1)[nonterminal] == q_star.argmax(1)[nonterminal]).float().mean()
)
entropy = float((-(probs * torch.log(probs.clamp_min(1e-8))).sum(1)).mean())
h = np.asarray(history)
print(
    f"agreement={agreement:.3f}, entropy={entropy:.3f}, "
    f"last clip fraction={h[-1, 1]:.3f}"
)
assert agreement >= 0.7 and math.isfinite(entropy)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(probs.T, aspect="auto", vmin=0, vmax=1)
axes[0].set_title("PPO policy")
axes[1].plot(h[:, 0], label="loss")
axes[1].plot(h[:, 1], label="clip fraction")
axes[1].legend()
fig.tight_layout()
plt.show()

음수 advantage action의 probability를 지나치게 낮추는 것도 surrogate 개선으로 과대평가될 수 있습니다. min 연산은 부호에 맞춰 양쪽의 과도한 ratio 변화를 제한합니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2, Eq. (1) and Eq. (6)`의 **policy-gradient/CPI surrogate**
- **핵심 식:**

  $$
  L^{CLIP}=\mathbb E_t\left[\min(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)\right]
  $$

- **입출력 shape:** `state [B] → policy [B, A], value [B]; logp/advantage/return [B]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `ProximalPolicyAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. old-policy importance ratio objective의 출발점을 계산한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 MuJoCo/Atari에서 parallel actors와 GAE를 사용한다. 축소판은 exact q_star advantage를 써 clipping mechanics를 격리한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.